# Dialectos en CSV

Los dialectos del módulo csv son un conjunto de reglas predefinidas o personalizadas que indican cómo está estructurado un archivo de texto separado por caracteres (delimitadores, comillas, saltos de línea, etc.).

Sirven para que se sepa exactamente cómo interpretar o generar un archivo CSV sin tener que configurar manualmente cada parámetro.

Cada dialecto es una clase o conjunto de propiedades que configuran estos parámetros:
* `delimiter`: El carácter que separa las columnas (ej. , ; \t).
___
* `quotechar`: El carácter que se usa para envolver textos que contienen el delimitador o saltos de línea (ej. comillas).
  
  Ejemplo: Una tabla con dos columnas: Nombre y Descripción. Si un registro contiene comas dentro del propio texto, el archivo CSV se rompería si no tuviera comillas.

  |Sin quotechar (Incorrecto)|Con quotechar usando comillas dobles " (Correcto)|
  |---|---|
  |Juan Pérez, Ingeniero, amante del café|Juan Pérez,"Ingeniero, amante del café"|
  |Se puede pensar que hay 3 columnas en lugar de 2|Todo lo que está dentro de " pertenece a una sola columna|
___
* `quoting`: CUÁNDO se deben usar las comillas (`QUOTE_ALL`, `QUOTE_MINIMAL`, `QUOTE_NONNUMERIC`, `QUOTE_NONE`).
  
| Constante | Comportamiento al Escribir (`writer`) | Comportamiento al Leer (`reader`) |
| :--- | :--- | :--- |
| **`csv.QUOTE_MINIMAL`** *(Por defecto)* | **Solo pone comillas si es estrictamente necesario**: cuando el campo contiene el delimitador (ej. una coma), el carácter de comilla o un salto de línea. | Lee las comillas de cualquier campo que las tenga y las elimina del resultado final. |
| **`csv.QUOTE_ALL`** | **Pone comillas a absolutamente todo**, sin importar si el campo es texto, un número, un booleano o si está vacío. | Igual que el anterior, procesa y limpia las comillas de todos los campos. |
| **`csv.QUOTE_NONNUMERIC`** | **Pone comillas a todo lo que NO sea un número** (textos, fechas, etc.). Los números (enteros y flotantes) se escriben sin comillas. | **Convierte automáticamente** los campos que no tienen comillas al tipo de dato `float` en Python. |
| **`csv.QUOTE_NONE`** | **Nunca pone comillas**. Si un campo contiene un delimitador, se está obligado a definir un carácter de escape (`escapechar`) para que no se rompa el formato. | No interpreta las comillas como contenedores de texto; las lee como si fueran parte del texto común. |
___
* `escapechar`: Un carácter para escapar delimitadores o comillas si aparecen dentro del texto. En el módulo csv de Python, el parámetro `escapechar` es una opción de configuración de los dialectos que define un carácter de escape de un solo carácter. Su función principal es quitarle el significado especial a otros caracteres del archivo (como el delimitador, las comillas o los saltos de línea) durante la lectura o la escritura. Por defecto, `escapechar` está establecido en None (desactivado).

**Funcionamiento al Escribir (csv.writer)**

Python necesita saber qué hacer si el texto de una celda contiene caracteres que coinciden con el delimitador (ej. ,) o con comillas (ej. ").
El comportamiento de `escapechar` depende estrechamente de la propiedad `quoting` (la política de entrecomillado) del dialecto:

**Escenario A: `quoting = csv.QUOTE_NONE` (Sin comillas)**

Si queremos que el archivo CSV no lleve comillas para envolver los textos, tenemos que definir un `escapechar` si los datos contienen delimitadores o saltos de línea. 

**Escenario B: `quoting = csv.QUOTE_MINIMAL, QUOTE_ALL, etc.` (Con comillas)**

Si el dialecto usa comillas, Python por defecto duplicará las comillas para escaparlas (ej. "Dijo ""hola""").

**Funcionamiento al Leer (csv.reader)**

Cuando se procesa un archivo, el `escapechar` le dice a Python: "Cualquier carácter que venga inmediatamente después de mí debe ser tratado como texto literal, no como una instrucción de formato".
* Si el lector encuentra \, (y \ es el escapechar), interpretará la coma como parte del texto de la celda, no como la separación hacia la siguiente columna.
* Si encuentra \n, interpretará el salto de línea como parte del texto de la celda, no como el fin de la fila/registro.
* El propio escapechar se elimina del resultado final que recibes en Python.
___
 
* `lineterminator`: Cómo se separan las filas al escribir (\r\n o \n).
___
* `skipinitialspace`: Si es `True`, ignora los espacios que vengan inmediatamente después del delimitador.

### Uso de `delimiter`, `quotechar`, `quoting` y `escapechar`

In [3]:
import csv

# Uso de delimitador ;
with open('recursos/dialestos01.csv', 'w', newline='', encoding='utf-8') as archivo:
    writer = csv.writer(archivo,delimiter=";")
    writer.writerow(['Nombre', 'Descripcion', 'Edad'])
    writer.writerow(['Ana', 'vive en Madrid', 30])
    writer.writerow(['Juan', 'vive en Barcelona, con su familia', 40])

# Uso de delimitador , pero hay una coma dentro de un campo de texto, por lo que ese campo estará entre comillas
with open('recursos/dialestos02.csv', 'w', newline='', encoding='utf-8') as archivo:
    writer = csv.writer(archivo,delimiter=",")
    writer.writerow(['Nombre', 'Descripcion', 'Edad'])
    writer.writerow(['Ana', 'vive en Madrid', 30])
    writer.writerow(['Juan', 'vive en Barcelona, con su familia', 40])  

# Uso de quotechar *
with open('recursos/dialestos03.csv', 'w', newline='', encoding='utf-8') as archivo:
    writer = csv.writer(archivo,quotechar='*')
    writer.writerow(['Nombre', 'Descripcion', 'Edad'])
    writer.writerow(['Ana', 'vive en Madrid', 30])
    writer.writerow(['Juan', 'vive en Barcelona, con su familia', 40]) # En este fila, el campo descripción irá entre comillas por contener el delimitador

# Uso de quotechar " y delimitador ; Es como el caso 2
with open('recursos/dialestos04.csv', 'w', newline='', encoding='utf-8') as archivo:
    writer = csv.writer(archivo,delimiter=",",quotechar='"')
    writer.writerow(['Nombre', 'Descripcion', 'Edad'])
    writer.writerow(['Ana', 'vive en Madrid', 30])
    writer.writerow(['Juan', 'vive en Barcelona, con su familia', 40]) # En este fila, el campo descripción no irá entre comillas porque no contiene el delimitador

# Uso de quoting con csv.MINIMAL
with open('recursos/dialestos05.csv', 'w', newline='', encoding='utf-8') as archivo:
    writer = csv.writer(archivo,quoting=csv.QUOTE_MINIMAL)
    writer.writerow(['Nombre', 'Descripcion', 'Edad'])
    writer.writerow(['Ana', 'vive en Madrid', 30])
    writer.writerow(['Juan', 'vive en Barcelona, con su familia', 40])

# Uso de quoting con csv.QUOTE_ALL
with open('recursos/dialestos06.csv', 'w', newline='', encoding='utf-8') as archivo:
    writer = csv.writer(archivo,quoting=csv.QUOTE_ALL)
    writer.writerow(['Nombre', 'Descripcion', 'Edad'])
    writer.writerow(['Ana', 'vive en Madrid', 30])
    writer.writerow(['Juan', 'vive en Barcelona, con su familia', 40])

# Uso de quoting con csv.QUOTE_ALL pero un campo contiene a su vez comillas
with open('recursos/dialestos07.csv', 'w', newline='', encoding='utf-8') as archivo:
    writer = csv.writer(archivo,quoting=csv.QUOTE_ALL)
    writer.writerow(['Nombre', 'Descripcion', 'Edad'])
    writer.writerow(['Ana', 'vive en "Madrid"', 30]) # Cambio ya que Madrid está en el texto entrecomillado
    writer.writerow(['Juan', 'vive en Barcelona, con su familia', 40])

# Uso de quoting con csv.QUOTE_NONNUMERIC
with open('recursos/dialestos08.csv', 'w', newline='', encoding='utf-8') as archivo:
    writer = csv.writer(archivo,quoting=csv.QUOTE_NONNUMERIC)
    writer.writerow(['Nombre', 'Descripcion', 'Edad'])
    writer.writerow(['Ana', 'vive en Madrid', 30])
    writer.writerow(['Juan', 'vive en Barcelona, con su familia', 40.5])

# Uso de quoting con csv.NONE por lo que obliga a usar escapechar
with open('recursos/dialestos09.csv', 'w', newline='', encoding='utf-8') as archivo:
    writer = csv.writer(archivo,quoting=csv.QUOTE_NONE,escapechar='\\')
    writer.writerow(['Nombre', 'Descripcion', 'Edad'])
    writer.writerow(['Ana', 'vive en "Madrid"', 30])
    writer.writerow(['Juan', 'vive en Barcelona, con su familia', 40.5])

### Uso de `lineterminator`

In [15]:
import csv

# lineterminator
with open('recursos/dialestos10.csv', 'w', newline='', encoding='utf-8') as archivo:
    writer = csv.writer(archivo,quoting=csv.QUOTE_ALL,lineterminator="\r\n")
    writer.writerow(['Nombre', 'Descripcion', 'Edad'])
    writer.writerow(['Ana', 'vive en "Madrid"', 30])
    writer.writerow(['Juan', 'vive en Barcelona, con su familia', 40])

with open('recursos/dialestos11.csv', 'w', newline='', encoding='utf-8') as archivo:
    writer = csv.writer(archivo,quoting=csv.QUOTE_ALL,lineterminator="\n")
    writer.writerow(['Nombre', 'Descripcion', 'Edad'])
    writer.writerow(['Ana', 'vive en "Madrid"', 30])
    writer.writerow(['Juan', 'vive en Barcelona, con su familia', 40])

### Uso de `skipinitialspace`

Para leer únicamente

In [15]:
# skipinitialspace
with open('recursos/datos-skipinitialspace.csv', 'r', newline='', encoding='utf-8') as archivo:
    lector = csv.reader(archivo,skipinitialspace=True)
    for linea in lector:
        print(linea)

print("="*150)

with open('recursos/datos-skipinitialspace.csv', 'r', newline='', encoding='utf-8') as archivo:
    lector = csv.reader(archivo,skipinitialspace=False)
    for linea in lector:
        print(linea)

['dni', 'nombre', 'edad', 'ciudad', 'telefono']
['75393127R', 'Cristina Hernández Ruiz', '47', 'Vigo', '946920448']
['59725148J', 'Elena Martínez Jiménez', '76', 'Bilbao', '754101479']
['26659883P', 'Isabel Jiménez González', '65', 'Valencia', '685493647']
['13419968C', 'Alejandro López Jiménez', '79', 'Vitoria', '941377996']
['dni', 'nombre', 'edad', 'ciudad', 'telefono']
['75393127R', 'Cristina Hernández Ruiz', '47', 'Vigo', '946920448']
['59725148J', '     Elena Martínez Jiménez', '76', 'Bilbao', '754101479']
['26659883P', 'Isabel Jiménez González', '65', '    Valencia', '685493647']
['13419968C', 'Alejandro López Jiménez', '79', 'Vitoria', '941377996']


#### **Hay dialectos predefinidos:**

`excel`
* Es el dialecto estándar para archivos compatibles con Microsoft Excel.
* Delimitador (delimiter): coma (,)
* Carácter de entrecomillado (quotechar): comilla doble (")
* Terminador de línea (lineterminator): \r\n
* Comportamiento: Solo pone comillas en los campos cuando es estrictamente necesario (QUOTE_MINIMAL).
  
`excel-tab`
* Idéntico al dialecto excel, pero adaptado para archivos separados por tabulaciones (muy común en archivos .tsv).
* Delimitador (delimiter): tabulación (\t)
  
`unix`
* Diseñado para el estándar de sistemas operativos tipo Unix.
* Delimitador (delimiter): coma (,)
* Carácter de entrecomillado (quotechar): comilla doble (")
* Terminador de línea (lineterminator): \n (salto de línea de Unix/Linux)
* Comportamiento: Pone comillas a absolutamente todos los campos (QUOTE_ALL).

In [17]:
import csv

# Usar el dialecto excel para escribir
with open('recursos/salida-excel.csv', 'w', newline='', encoding='utf-8') as archivo:
    writer = csv.writer(archivo, dialect='excel')
    writer.writerow(['Nombre', 'Ciudad', 'Edad'])
    writer.writerow(['Ana', 'Madrid', '30'])

# Usar el dialecto excel-tab para escribir
with open('recursos/salida-excel-tab.csv', 'w', newline='', encoding='utf-8') as archivo:
    writer = csv.writer(archivo, dialect='excel-tab')
    writer.writerow(['Nombre', 'Ciudad', 'Edad'])
    writer.writerow(['Ana', 'Madrid', '30'])

# Usar el dialecto unix para escribir
with open('recursos/salida-unix.csv', 'w', newline='', encoding='utf-8') as archivo:
    writer = csv.writer(archivo, dialect='unix')
    writer.writerow(['Nombre', 'Ciudad', 'Edad'])
    writer.writerow(['Ana', 'Madrid', '30'])

    print(f"delimiter {writer.dialect.delimiter}")
    print(f"quotechar {writer.dialect.quotechar}")
    print(f"quoting {writer.dialect.quoting}")
    print(f"escapechar {writer.dialect.escapechar}")
    print(f"lineterminator {writer.dialect.lineterminator}")
    print(f"skipinitialspace {writer.dialect.skipinitialspace}")

delimiter ,
quotechar "
quoting 1
escapechar None
lineterminator 

skipinitialspace False


## Listado de dialectos, registrar dialectos
`register_dialect`, `list_dialects`, `get_dialect` y `unregister_dialect`

In [18]:
import csv

# Se registra un dialecto personalizado
csv.register_dialect(
    'mi_dialecto',
    delimiter=';',
    quotechar='"',
    quoting=csv.QUOTE_ALL
)

# Usado para escribir
with open('recursos/salida-mi_dialecto.csv', 'w', newline='',  encoding='utf-8') as archivo:
    escritor = csv.writer(archivo, dialect='mi_dialecto')
    escritor.writerow(['Nombre', 'Edad', 'Ciudad'])
    escritor.writerow(['Ana', 30, 'Madrid'])
    escritor.writerow(['Carlos', 25, 'Barcelona'])

# Usado par leer
with open('recursos/salida-mi_dialecto.csv', 'r', newline='', encoding='utf-8') as archivo:
    lector = csv.reader(archivo,dialect='mi_dialecto')
    for linea in lector:
        print(f"{linea}")

    
# Lista de dialectos
dialectos = csv.list_dialects()
print("="*150,"\nLista de dialectos")
for i in dialectos:
    print("* ",i)

print("="*150)

# Obtener un dialecto
dialecto = csv.get_dialect("excel")
print(f"delimiter {dialecto.delimiter}")
print(f"quotechar {dialecto.quotechar}")
print(f"quoting {dialecto.quoting}")
print(f"escapechar {dialecto.escapechar}")
print(f"lineterminator {dialecto.lineterminator}")
print(f"skipinitialspace {dialecto.skipinitialspace}")

print("="*150)

# Quitar un dialecto
csv.unregister_dialect('mi_dialecto')
dialectos = csv.list_dialects()
print("Lista de dialectos despues de eliminar el personalizado")
for i in dialectos:
    print("* ",i)

['Nombre', 'Edad', 'Ciudad']
['Ana', '30', 'Madrid']
['Carlos', '25', 'Barcelona']
Lista de dialectos
*  excel
*  excel-tab
*  unix
*  mi_dialecto
delimiter ,
quotechar "
quoting 0
escapechar None
lineterminator 

skipinitialspace False
Lista de dialectos despues de eliminar el personalizado
*  excel
*  excel-tab
*  unix


## Detectar un dialecto
La clase `Sniffer`
* `sniff`: Analiza una muestra y devuelve una objeto `dialect`.
* `has_header`: Comprueba si hay cabecera en la muestra.

In [5]:
import csv

#fichero = "recursos/datos.csv"
fichero = "recursos/datos-sincabecera.csv"

with open(fichero, 'r', encoding='utf-8') as archivo:
    muestra = archivo.read(1024) # read no pertenece a csv, es un read normal y corriente

    # El Sniffer deduce el dialecto
    dialecto_detectado = csv.Sniffer().sniff(muestra)

    if csv.Sniffer().has_header(muestra):
        print("Tiene cabecera")
    else:
        print("No tiene cabecera")
        
    print(f"delimiter {dialecto_detectado.delimiter}")
    print(f"quotechar {dialecto_detectado.quotechar}")
    print(f"quoting {dialecto_detectado.quoting}")
    print(f"escapechar {dialecto_detectado.escapechar}")
    print(f"lineterminator {dialecto_detectado.lineterminator}")
    print(f"skipinitialspace {dialecto_detectado.skipinitialspace}")

No tiene cabecera
delimiter ,
quotechar "
quoting 0
escapechar None
lineterminator 

skipinitialspace False
